# 08 — Within-Repo Robustness + Publication Figures  
### *Coding Alone* · ICSE 2027 SEIS pipeline

**Self-contained.** Set PROJECT_ID → **Runtime → Run all**. Reads your metric tables (cheap).

Two jobs:
1. **Within-repo fixed-effects trends** — the composition-killer. Do the isolation signals move *within the same projects* over time, or is the ecosystem trend just a changing repo mix? Repo fixed effects (demeaning) + cluster-robust SEs answer this directly.
2. **Publication figures** — clean, colorblind-safe (Okabe–Ito) trend figure + NRR-by-year, saved as PDF+PNG for the manuscript.


## Config

In [ ]:
PROJECT_ID = "YOUR-GCP-PROJECT-ID"   # <-- edit
BQ_DATASET="seis"; BQ_LOCATION="US"; DRIVE_BASE="/content/drive/MyDrive/ICSE_SEIS"
WIN_START="201801"; WIN_END="202506"; MDI_END="202412"; GAP=["202110"]
MIN_MONTHS=24; SAMPLE_REPOS=8000
assert PROJECT_ID!="PASTE-YOUR-PROJECT-ID-HERE","Set PROJECT_ID first."


## Setup

In [ ]:
import numpy as np, pandas as pd, os
import statsmodels.api as sm
import matplotlib.pyplot as plt
from google.colab import auth, drive
from google.cloud import bigquery
auth.authenticate_user(); client=bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
drive.mount('/content/drive'); os.makedirs(DRIVE_BASE+"/figures", exist_ok=True); os.makedirs(DRIVE_BASE+"/tables", exist_ok=True)
P=f"{PROJECT_ID}.{BQ_DATASET}"
def q(s): return client.query(s).result().to_dataframe()
def ymidx(s): return (s.str[:4].astype(int)-2018)*12+(s.str[4:].astype(int)-1)
OKABE={"blue":"#0072B2","orange":"#E69F00","green":"#009E73","red":"#D55E00","purple":"#CC79A7"}
print("Ready.", P)


## 1. Within-repo fixed-effects trends
Sample repos with ≥24 active months, demean each metric and time *within repo* (repo fixed effects), then regress. The `within_slope` is the average trend **inside** projects. If its sign matches the ecosystem trend and it's significant, composition is ruled out.

In [ ]:
panel = q(f"""
WITH act AS (
  SELECT repo_id, COUNTIF(active) m FROM `{P}.metrics_panel`
  WHERE ym BETWEEN '{WIN_START}' AND '{WIN_END}' AND ym NOT IN UNNEST({GAP})
  GROUP BY repo_id),
samp AS (SELECT repo_id FROM act WHERE m>={MIN_MONTHS}
         ORDER BY FARM_FINGERPRINT(CAST(repo_id AS STRING)) LIMIT {SAMPLE_REPOS})
SELECT p.repo_id, p.ym, IF(p.active,p.cd,NULL) cd, p.swr, p.reciprocity cr, p.hie_norm hie
FROM `{P}.metrics_panel` p JOIN samp USING(repo_id)
WHERE p.ym BETWEEN '{WIN_START}' AND '{WIN_END}' AND p.ym NOT IN UNNEST({GAP})""")
panel["t"]=ymidx(panel.ym)
print("panel:", panel.shape, "| repos:", panel.repo_id.nunique())

def within_trend(metric):
    d=panel.dropna(subset=[metric])[["repo_id","t",metric]].copy()
    d["y"]=d[metric]-d.groupby("repo_id")[metric].transform("mean")
    d["x"]=d["t"]-d.groupby("repo_id")["t"].transform("mean")
    m=sm.OLS(d["y"], sm.add_constant(d["x"])).fit(cov_type="cluster", cov_kwds={"groups":d["repo_id"]})
    return {"metric":metric, "within_slope_per_month":round(m.params["x"],6),
            "p_value":round(m.pvalues["x"],5), "n_repos":d.repo_id.nunique(), "n_obs":len(d)}
within=pd.DataFrame([within_trend(x) for x in ["cd","swr","cr","hie"]])
within.to_csv(f"{DRIVE_BASE}/tables/within_repo_trends.csv", index=False)
within


## 2. NRR within-repo (newcomer-weighted, repo FE)
Newcomer response rate trend within repos that host newcomers across years. Sparser, so we require repos with newcomers in ≥3 distinct years.

In [ ]:
nrr_panel = q(f"""
WITH ry AS (
  SELECT repo_id, ym, AVG(IF(responses>0,1.0,0)) nrr, COUNT(*) nc
  FROM `{P}.mdi_newcomers`
  WHERE ym BETWEEN '{WIN_START}' AND '{MDI_END}' AND ym NOT IN UNNEST({GAP})
  GROUP BY repo_id, ym),
elig AS (SELECT repo_id FROM ry GROUP BY repo_id HAVING COUNT(DISTINCT SUBSTR(ym,1,4))>=3)
SELECT r.* FROM ry r JOIN elig USING(repo_id)""")
nrr_panel["t"]=ymidx(nrr_panel.ym)
d=nrr_panel.copy()
d["y"]=d.nrr-d.groupby("repo_id").nrr.transform("mean")
d["x"]=d.t-d.groupby("repo_id").t.transform("mean")
m=sm.WLS(d["y"], sm.add_constant(d["x"]), weights=d["nc"]).fit(cov_type="cluster", cov_kwds={"groups":d["repo_id"]})
print(f"NRR within-repo slope/month: {m.params['x']:.6f}  (p={m.pvalues['x']:.5f}, repos={d.repo_id.nunique()})")


## 3. Publication figure — the four repo-month signals + NRR

In [ ]:
eco = q(f"""
SELECT ym, AVG(IF(active,cd,NULL)) cd, AVG(swr) swr, AVG(reciprocity) cr, AVG(hie_norm) hie
FROM `{P}.metrics_panel` WHERE ym BETWEEN '{WIN_START}' AND '{WIN_END}' AND ym NOT IN UNNEST({GAP})
GROUP BY ym ORDER BY ym""")
nrr = q(f"""SELECT ym, AVG(IF(responses>0,1.0,0)) nrr FROM `{P}.mdi_newcomers`
  WHERE ym BETWEEN '{WIN_START}' AND '{MDI_END}' AND ym NOT IN UNNEST({GAP}) GROUP BY ym ORDER BY ym""")
eco=eco.merge(nrr,on="ym",how="left"); eco["date"]=pd.to_datetime(eco.ym+"01",format="%Y%m%d")

plt.rcParams.update({"font.size":11,"axes.spines.top":False,"axes.spines.right":False})
fig,ax=plt.subplots(1,5,figsize=(18,3.4))
specs=[("swr","Solitary Workflow Ratio","blue"),("hie","Human Interaction Entropy","green"),
       ("cr","Communication Reciprocity","orange"),("nrr","Newcomer Response Rate","purple"),
       ("cd","Collaboration Density","red")]
covid=pd.Timestamp("2020-03-01"); ai=pd.Timestamp("2022-11-01")
for a,(col,title,ckey) in zip(ax,specs):
    d=eco.dropna(subset=[col])
    a.plot(d.date,d[col],color=OKABE[ckey],lw=1.6)
    a.axvline(covid,ls=":",color="grey",lw=1); a.axvline(ai,ls="--",color="black",lw=1)
    a.set_title(title,fontsize=10.5); a.grid(alpha=.25)
fig.suptitle("Socio-technical isolation signals across ~50k GitHub repositories, 2018–2025 "
             "(dotted = COVID onset, dashed = ChatGPT release)", y=1.06, fontsize=11)
fig.tight_layout()
for ext in ("pdf","png"):
    fig.savefig(f"{DRIVE_BASE}/figures/fig1_trends.{ext}", dpi=200, bbox_inches="tight")
plt.show(); print("Saved fig1_trends.pdf / .png")


## Done ✅
- `within_repo_trends.csv` + the printed NRR within-repo slope: **the composition check**. Signs matching the ecosystem trends (SWR +, HIE −, CR −, NRR −) with small p-values = isolation is happening *inside* projects, not just from a changing repo mix.
- `figures/fig1_trends.pdf/png`: the manuscript's main descriptive figure.

If within-repo confirms the ecosystem trends, the descriptive claim is airtight and we move to drafting the manuscript. Send me the `within` table and the NRR within-repo line.